# Part 3: Baseline and model evaluation

Owner: _(name)_

Brief, section 5: compare all three models with a last-observed-value baseline on identical test
dates and the same one-day horizon; MAE and RMSE in index points; actual-versus-predicted plots and an
error comparison table; residual patterns, overfitting, computational cost and limitations; say
whether any improvement is meaningful. A well-supported "nothing beats the baseline" is a valid finding.

Each model notebook saves its test-set predictions (already converted to index points) to
`results/<model>_predictions.csv` plus a `<model>_meta.json` with settings and training time.

### Setup (same cell in every notebook)

Runs in Colab and in Cursor. In Colab it clones the repository first; for a private repository
paste a GitHub token when asked (Settings > Developer settings > Fine-grained tokens, read access
to this repository is enough).

In [ ]:
import sys, pathlib

REPO_URL = "https://github.com/Makkaii7/sp500-forecasting.git"   # <- set once after the repo exists

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not pathlib.Path("sp500-forecasting").exists() and not pathlib.Path("src/config.py").exists():
    from getpass import getpass
    token = getpass("GitHub token (leave empty if the repo is public): ")
    url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
    import subprocess
    subprocess.run(["git", "clone", "--quiet", url], check=True)
if IN_COLAB and pathlib.Path("sp500-forecasting").exists():
    import os
    os.chdir("sp500-forecasting")

root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src import config, common

df = common.load_prices()
y = common.target_series(df)
train, val, test = common.split(y)
print(root)
print(df.index.min().date(), "to", df.index.max().date(), "|", len(df), "trading days")
print(f"train {train.index.min().date()} to {train.index.max().date()} ({len(train)}) | validation {val.index.min().date()} to {val.index.max().date()} ({len(val)}) | test {test.index.min().date()} to {test.index.max().date()} ({len(test)})")
df.tail(3)

## 1. The baseline: tomorrow's close = today's close

In [ ]:
baseline = common.last_value_baseline(test.index, df)
actual = common.actual_price(test.index, df)
# saved like a model, so it appears in the same table and plots; its "target" prediction is a zero return
common.save_predictions("baseline_last_value", df, pd.Series(0.0, index=test.index), settings={"rule": "close_t = close_{t-1}"}, train_seconds=0.0)
print("baseline MAE  (index points)", round(common.mae(actual, baseline), 2))
print("baseline RMSE (index points)", round(common.rmse(actual, baseline), 2))

## 2. Error comparison table: every model in `results/`, identical test dates

In [ ]:
score = common.scorecard()
base_rmse = score.loc["baseline_last_value", "RMSE (index points)"]
score["RMSE vs baseline (%)"] = ((score["RMSE (index points)"] / base_rmse - 1) * 100).round(2)
score

## 3. Actual versus predicted, one panel per model

In [ ]:
import glob
files = sorted(glob.glob(str(root / "results" / "*_predictions.csv")))
fig, axes = plt.subplots(len(files), 1, figsize=(11, 3.2 * len(files)), sharex=True)
for ax, f in zip(np.atleast_1d(axes), files):
    p = pd.read_csv(f, parse_dates=[0], index_col=0)
    p["actual_close"].plot(ax=ax, alpha=0.6, label="actual close")
    p["predicted_close"].plot(ax=ax, alpha=0.9, label="predicted close")
    ax.set_title(pathlib.Path(f).stem.replace("_predictions", "")); ax.set_ylabel("index points"); ax.legend()
plt.tight_layout()

## 4. Residual patterns

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf
fig, axes = plt.subplots(len(files), 2, figsize=(12, 3 * len(files)))
for row, f in zip(np.atleast_2d(axes), files):
    p = pd.read_csv(f, parse_dates=[0], index_col=0)
    resid = (p["actual_close"] - p["predicted_close"]).dropna()
    resid.plot(ax=row[0], title=pathlib.Path(f).stem.replace("_predictions", "") + ": residuals (index points)")
    plot_acf(resid, lags=20, ax=row[1], title="ACF of residuals")
plt.tight_layout()

## 5. Is any improvement meaningful?

Two models can differ by a few index points of RMSE for reasons that are pure chance. Before calling
a difference real, look at: its size relative to the baseline (the % column), whether it holds in both
halves of the test period (below), and whether the model's residuals show any structure the baseline's
do not. Write the conclusion in words a reader can check against the table.

In [ ]:
half = len(test) // 2
rows = []
for f in files:
    p = pd.read_csv(f, parse_dates=[0], index_col=0)
    name = pathlib.Path(f).stem.replace("_predictions", "")
    rows.append({"model": name,
                 "RMSE first half": common.rmse(p["actual_close"].iloc[:half], p["predicted_close"].iloc[:half]),
                 "RMSE second half": common.rmse(p["actual_close"].iloc[half:], p["predicted_close"].iloc[half:])})
pd.DataFrame(rows).set_index("model").round(2)

## 6. Computational cost and limitations

Use the "train seconds" column from the scorecard and the settings in each `<model>_meta.json`.
Discuss: training time against accuracy, sensitivity to the lookback window and to the random seed,
the single test period, and what the one-day horizon does and does not tell you.